# LRU Cache

## Problem Description
Design a data structure that follows the constraints of a **Least Recently Used (LRU) Cache**.

Implement the `LRUCache` class with two methods:
- `get(key)`: Return the value of the key if the key exists, otherwise return -1. This is a "use" operation.
- `put(key, value)`: Update the value of the key if the key exists. Otherwise, add the key-value pair. If the number of keys exceeds capacity, evict the least recently used key.

**Both operations must run in O(1) average time!**

## Example
```python
lru = LRUCache(2)
lru.put(1, 1)      # cache: {1: 1}
lru.put(2, 2)      # cache: {1: 1, 2: 2}
print(lru.get(1))  # returns 1, cache: {2: 2, 1: 1} (1 moved to most recent)
lru.put(3, 3)      # cache: {1: 1, 3: 3} (2 evicted as LRU)
print(lru.get(2))  # returns -1 (not found)
lru.put(4, 4)      # cache: {3: 3, 4: 4} (1 evicted as LRU)
print(lru.get(1))  # returns -1
print(lru.get(3))  # returns 3
print(lru.get(4))  # returns 4
```

## Complexity
- **get()**: O(1) average time
- **put()**: O(1) average time
- **Space**: O(capacity)

## Why Uber cares
- **Caching**: Frequently accessed data (locations, driver info, ratings)
- **Real-time systems**: Need fast access + memory constraints
- **System design**: Demonstrates both algorithmic and design thinking
- **Interview filter**: Tests if you can implement composite data structures under pressure

## Approach: HashMap + Doubly Linked List

### Why this combination?
1. **HashMap**: O(1) lookups by key
2. **Doubly Linked List**: O(1) insertion/deletion if you have pointer to node
3. **Order tracking**: List maintains LRU order (least recent at head, most recent at tail)

### Key Insight:
- Most Recently Used → Move to tail
- Least Recently Used → At head
- Eviction → Remove head node

### Data Structure:
```
HashMap: {key → DLLNode}

DLL (Most Recently Used ←→ Least Recently Used):
dummy_head ↔ [node1] ↔ [node2] ↔ ... ↔ [nodeN] ↔ dummy_tail
             (LRU)                            (MRU)
```

In [ ]:
class DNode:
    """Doubly Linked List Node"""
    def __init__(self, key=0, value=0):
        self.key = key
        self.value = value
        self.prev = None
        self.next = None

class LRUCache:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.cache = {}  # key → DNode
        
        # Dummy head and tail for easier insertion/deletion
        self.head = DNode()  # Most recently used side
        self.tail = DNode()  # Least recently used side
        self.head.next = self.tail
        self.tail.prev = self.head
    
    def get(self, key: int) -> int:
        """Get value and mark as recently used"""
        if key not in self.cache:
            print(f"❌ get({key}): not found, return -1")
            return -1
        
        node = self.cache[key]
        self._move_to_end(node)  # Mark as most recently used
        print(f"✅ get({key}): found {node.value}, moved to MRU")
        return node.value
    
    def put(self, key: int, value: int) -> None:
        """Put key-value pair, mark as recently used"""
        if key in self.cache:
            # Update existing key
            node = self.cache[key]
            node.value = value
            self._move_to_end(node)
            print(f"📝 put({key}, {value}): updated existing, moved to MRU")
        else:
            # Add new key
            if len(self.cache) >= self.capacity:
                # Evict LRU (first node after head)
                lru_node = self.head.next
                self._remove_node(lru_node)
                del self.cache[lru_node.key]
                print(f"🗑️  Evicted LRU key={lru_node.key}")
            
            node = DNode(key, value)
            self._add_to_end(node)
            self.cache[key] = node
            print(f"➕ put({key}, {value}): added new, cache size={len(self.cache)}")
    
    def _move_to_end(self, node):
        """Move node to end (most recently used)"""
        self._remove_node(node)
        self._add_to_end(node)
    
    def _remove_node(self, node):
        """Remove node from list"""
        node.prev.next = node.next
        node.next.prev = node.prev
    
    def _add_to_end(self, node):
        """Add node to end (before tail, as MRU)"""
        node.next = self.tail
        node.prev = self.tail.prev
        self.tail.prev.next = node
        self.tail.prev = node

## Test Cases

In [ ]:
# Example 1
print("Example 1: Capacity = 2")
lru = LRUCache(2)
lru.put(1, 1)
lru.put(2, 2)
print(f"get(1) = {lru.get(1)}")
lru.put(3, 3)  # Evict 2
print(f"get(2) = {lru.get(2)}")
lru.put(4, 4)  # Evict 1
print(f"get(1) = {lru.get(1)}")
print(f"get(3) = {lru.get(3)}")
print(f"get(4) = {lru.get(4)}")

print("\n" + "="*50 + "\n")

# Example 2
print("Example 2: Capacity = 1")
lru2 = LRUCache(1)
lru2.put(0, 0)
print(f"get(0) = {lru2.get(0)}")
lru2.put(1, 1)  # Evict 0
print(f"get(0) = {lru2.get(0)}")

print("\n" + "="*50 + "\n")

# Example 3: Multiple operations
print("Example 3: Complex operations")
lru3 = LRUCache(3)
lru3.put(1, "apple")
lru3.put(2, "banana")
lru3.put(3, "cherry")
print(f"get(1) = {lru3.get(1)}")
lru3.put(4, "date")  # Evict 2 (LRU)
print(f"get(2) = {lru3.get(2)}")
lru3.put(5, "elderberry")  # Evict 3 (LRU now)
print(f"get(3) = {lru3.get(3)}")
lru3.put(1, "apple_v2")  # Update 1
print(f"get(1) = {lru3.get(1)}")

## Pointer Manipulation Details

### Adding to end (before tail):
```
Before: ... ↔ tail.prev ↔ tail

After:  ... ↔ tail.prev ↔ node ↔ tail

Code:
  node.next = tail              # node → tail
  node.prev = tail.prev         # node ← tail.prev
  tail.prev.next = node         # tail.prev → node
  tail.prev = node              # tail ← node
```

### Removing node:
```
Before: A ↔ B ↔ C

After:  A ↔ C (B removed)

Code:
  B.prev.next = B.next     # A → C
  B.next.prev = B.prev     # C ← A
```

## Interview Tips

1. **Draw the structure**: Diagram the DLL with dummy nodes
2. **Test edge cases**: 
   - Capacity = 1
   - Update existing keys
   - Accessing after eviction
3. **Explain pointer logic**: Be clear about why dummy nodes help
4. **Time complexity**: Emphasize O(1) all operations
5. **Real-world context**: Mention caching, CPU cache eviction policies

## Alternatives Discussed in Interview
- **OrderedDict** (Python 3.7+): Maintains insertion order, simpler but less educational
- **Using just HashMap**: Can't achieve O(1) eviction
- **Using array + sorting**: Would be O(n log n) for eviction